# Merge Products and Product Details Tables
Creates a unified table with XML-formatted product information.

**Source Tables:**
- `llmagent.dev.products` - Product metadata
- `llmagent.dev.product_details` - Product documentation

**Target Table:**
- `llmagent.dev.product_merged` - Combined data with XML format

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws

spark = SparkSession.builder.appName("MergeProductTables").getOrCreate()

# Configuration
PRODUCTS_TABLE = "llmagent.dev.products"
DETAILS_TABLE = "llmagent.dev.product_details"
MERGED_TABLE = "llmagent.dev.product_merged"

print(f"Source Tables:")
print(f"  Products: {PRODUCTS_TABLE}")
print(f"  Details: {DETAILS_TABLE}")
print(f"Target Table: {MERGED_TABLE}")

In [0]:
# Read both tables
df_products = spark.read.table(PRODUCTS_TABLE)
df_details = spark.read.table(DETAILS_TABLE)

print(f"Products table: {df_products.count()} rows")
print(f"Details table: {df_details.count()} rows")

print("\nProducts Schema:")
df_products.printSchema()

print("\nDetails Schema:")
df_details.printSchema()

In [0]:
# Join tables on product_name
df_merged = df_products.join(
    df_details,
    on="product_name",
    how="left"
)

print(f"✓ Joined tables: {df_merged.count()} rows")
df_merged.printSchema()

In [0]:
# Create XML formatted column
# Format: <product_name>value</product_name><product_category>value</product_category>...
df_with_xml = df_merged.select(
    "*",
    concat_ws(
        "",
        concat_ws("", col("product_name").cast("string").alias("_temp1")),
    ).alias("_temp_skip")
)

# Using SQL for better control over XML formatting
df_merged.createOrReplaceTempView("merged_temp")

In [0]:
# Create the merged table with XML column using SQL
spark.sql(f"""
    CREATE OR REPLACE TABLE {MERGED_TABLE} AS
    SELECT
        product_id,
        product_name,
        product_category,
        product_sub_category,
        product_doc,
        CONCAT(
            '<product_name>', product_name, '</product_name>',
            '<product_category>', product_category, '</product_category>',
            '<product_sub_category>', product_sub_category, '</product_sub_category>',
            '<product_doc>', 
                COALESCE(product_doc, ''),
            '</product_doc>'
        ) AS product_xml
    FROM merged_temp
""")

print(f"✓ Created table {MERGED_TABLE}")

In [0]:
# Verify the table
df_result = spark.read.table(MERGED_TABLE)
print(f"✓ Total rows: {df_result.count()}")

print("\nTable Schema:")
df_result.printSchema()

In [0]:
# Show sample data
display(spark.sql(f"""
    SELECT
        product_id,
        product_name,
        product_category,
        product_sub_category,
        LENGTH(product_doc) as doc_length,
        SUBSTR(product_xml, 1, 300) as xml_preview
    FROM {MERGED_TABLE}
    LIMIT 5
"""))

In [0]:
# Show full XML example
display(spark.sql(f"""
    SELECT
        product_id,
        product_name,
        product_xml
    FROM {MERGED_TABLE}
    LIMIT 1
"""))

In [0]:
# Summary statistics
display(spark.sql(f"""
    SELECT
        COUNT(*) as total_records,
        COUNT(DISTINCT product_category) as unique_categories,
        COUNT(DISTINCT product_sub_category) as unique_sub_categories,
        COUNT(CASE WHEN product_doc IS NOT NULL THEN 1 END) as records_with_docs,
        MIN(LENGTH(product_xml)) as min_xml_length,
        MAX(LENGTH(product_xml)) as max_xml_length,
        ROUND(AVG(LENGTH(product_xml)), 0) as avg_xml_length
    FROM {MERGED_TABLE}
"""))

In [0]:
%sql

select * from llmagent.dev.product_merged
limit 10

In [0]:
%sql
ALTER TABLE llmagent.dev.product_merged
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)